# Phase 2 - Text Preprocessing

Goal: take the 2M stratified sample from Phase 1 and produce a `tokens` column ready for TF-IDF / Word2Vec, plus a canonicalized `label_canonical` column that collapses ALL-CAPS historical labels into Title-Case 2020+ names.

Phases 0 and 1 must be passing. Phase 1's `sample_2m.parquet` must be on Drive.

## Cell 1 - Bootstrap (drive + repo + spark + nltk data)

In [ ]:
# project repo
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# pull latest from repo
import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

# install deps if not already present (cheap if already installed)
!pip install -r /content/project/requirements-train.txt -q

# java 11
!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# pre-download nltk data (stopwords, wordnet, punkt). this writes to a project
# directory we can later commit so streamlit cloud doesnt need to download.
import nltk
nltk_dir = '/content/project/dashboard/assets/nltk_data'
os.makedirs(nltk_dir, exist_ok=True)
nltk.data.path.insert(0, nltk_dir)
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    try:
        nltk.download(pkg, download_dir=nltk_dir, quiet=True)
    except Exception as e:
        print(f'warning: nltk download for {pkg} failed: {e}')
print('nltk data ready')

# spark up
from src.spark_setup import get_spark
spark = get_spark(app_name='phase2-preprocess')
print('spark', spark.version, 'ready')

## Cell 2 - Load the Phase 1 sample

We read straight from Drive. If this fails, Phase 1 didn't finish.

In [ ]:
from pyspark.sql import functions as F

in_path = '/content/drive/MyDrive/cs6513/sample_2m.parquet'
df = spark.read.parquet(in_path)
print(f'loaded {df.count():,} rows')
df.printSchema()

## Cell 3 - Inspect raw label distribution

This is the issue we discovered after Phase 1. Look for ALL-CAPS labels (HEATING, PLUMBING, etc.) sitting next to their Title Case counterparts.

In [ ]:
raw_top = df.groupBy('problem').count().orderBy(F.desc('count')).limit(30).toPandas()
print('top 30 raw labels:')
print(raw_top.to_string(index=False))
print(f'\ndistinct raw labels: {df.select("problem").distinct().count()}')

## Cell 4 - Apply label canonicalization

Collapse known synonym pairs (HEATING -> Heat/Hot Water, etc.) using the map in `src.preprocess.LABEL_CANONICAL_MAP`. Anything not in the map passes through with its original spelling preserved.

In [ ]:
from src.preprocess import add_canonical_label

df_labeled = add_canonical_label(df, in_col='problem', out_col='label_canonical')

canonical_top = (
    df_labeled.groupBy('label_canonical')
    .count()
    .orderBy(F.desc('count'))
    .limit(30)
    .toPandas()
)
print('top 30 canonical labels:')
print(canonical_top.to_string(index=False))
print(f'\ndistinct canonical labels: {df_labeled.select("label_canonical").distinct().count()}')

## Cell 5 - Tokenize, stopword, lemmatize

TextPreprocessor is a Spark Transformer that wraps NLTK behavior into a UDF and adds a `tokens` column. We run it on `problem_detail` (the descriptor field, the actual free text).

In [ ]:
from src.preprocess import TextPreprocessor

preproc = TextPreprocessor(input_col='problem_detail', output_col='tokens')
df_tok = preproc.transform(df_labeled)

# show a quick before/after on 5 rows so we can sanity check the tokenization
df_tok.select('problem_detail', 'tokens').show(5, truncate=80)

## Cell 6 - Token quality checks

We want to know:
1. How many rows have an empty token list (description was null or all-stopwords)
2. The distribution of token counts per row
3. The most common tokens overall (sanity check that stopwords are gone)

In [ ]:
# rows with empty token lists
n_empty = df_tok.filter(F.size('tokens') == 0).count()
n_total = df_tok.count()
print(f'rows with empty token list: {n_empty:,} / {n_total:,} ({100 * n_empty / n_total:.1f}%)')

# token count distribution
df_tok.select(F.size('tokens').alias('n_tokens')).describe().show()

# top 30 tokens overall
top_tokens = (
    df_tok.select(F.explode('tokens').alias('tok'))
    .groupBy('tok')
    .count()
    .orderBy(F.desc('count'))
    .limit(30)
    .toPandas()
)
print('top 30 tokens after preprocessing:')
print(top_tokens.to_string(index=False))

## Cell 7 - Write the preprocessed parquet

Phase 3 (classifier training) reads from this file. We keep the raw `problem` column too in case any downstream phase wants the original string.

In [ ]:
out_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
(
    df_tok
    # keep only columns we need downstream - dropping resolution_description
    # since its post-resolution boilerplate that leaks the answer
    .select(
        'unique_key', 'created_date', 'closed_date',
        'agency', 'problem', 'label_canonical', 'problem_detail',
        'borough', 'incident_zip', 'latitude', 'longitude',
        'status', 'tokens',
    )
    .write.mode('overwrite')
    .parquet(out_path)
)
print(f'preprocessed parquet written to {out_path}')

# also commit nltk data + a quick stats file to git so streamlit cloud has it
import json
stats = {
    'rows_total': n_total,
    'rows_empty_tokens': n_empty,
    'distinct_canonical_labels': int(df_tok.select('label_canonical').distinct().count()),
}
with open('/content/project/dashboard/assets/preprocess_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print('stats saved to dashboard/assets/preprocess_stats.json')

## Cell 8 - Push the new artifacts to GitHub

We commit `nltk_data/` (~30 MB, fine for GitHub) and `preprocess_stats.json` so Streamlit Cloud has them at deploy time. Models and parquets stay on Drive (gitignored).

If you havent set up git credentials inside Colab, this cell will skip with a friendly message.

In [ ]:
import subprocess

try:
    subprocess.run(
        ['git', '-C', '/content/project', 'add', 'dashboard/assets/'],
        check=True
    )
    res = subprocess.run(
        ['git', '-C', '/content/project', 'commit', '-m', 'phase 2: nltk data + preprocess stats'],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print('committed; you can push manually or i (claude) will push from local')
    else:
        print('nothing new to commit (probably already there)')
        print(res.stdout)
        print(res.stderr)
except Exception as e:
    print(f'git commit skipped: {e}')
    print('not a problem - claude will pick up the changes from drive locally')

## Phase 2 - Done when

- Cell 4 shows canonical labels with `Heat/Hot Water` count = sum of HEATING + Heat/Hot Water raw counts (probably ~150K combined). The ALL-CAPS variants should NOT appear in the canonical top-30.
- Cell 6 reports <2% empty token rows.
- Cell 6 top-30 tokens are domain-specific (`tree`, `noise`, `parked`, `loud`, `garbage`, `apartment`, `light`, `sidewalk`, etc.) with no English stopwords.
- Cell 7 writes `sample_2m_preprocessed.parquet` to Drive.

Save the notebook back to GitHub and we move to Phase 3 (classifier training).